# Assignment 05

## Spark Fundamentals and Data Processing using PySpark

### Objective

To understand Spark fundamentals and perform data cleaning, transformation, filtering, aggregation, schema modification, and build a complete data processing pipeline using PySpark DataFrames.

In [2]:
import os

os.environ["PYSPARK_PYTHON"] = r"C:\Users\prapt\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\prapt\AppData\Local\Programs\Python\Python310\python.exe"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Assignment_05")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

In [3]:
data = [
    (1, "Alice", "alice@gmail.com", "2024-07-01", 25, "Premium", "West", "Electronics", "Delhi", 500, "Active", 101),
    (2, "Bob", None, "2024-07-01", 30, "Basic", "East", "Clothing", "Mumbai", 200, None, 102),
    (3, "Charlie", "charlie@gmail.com", "2024-07-02", 22, "Premium", "West", "Electronics", "Delhi", 700, "Active", 101),
    (4, "David", "david@gmail.com", "2024-07-02", 35, "Premium", "North", "Furniture", "Chennai", None, "Inactive", 103),
    (5, "Eva", "", "2024-07-03", 28, "Basic", "West", "Clothing", "Delhi", 400, None, 101),
    (5, "Eva", "", "2024-07-03", 28, "Basic", "West", "Clothing", "Delhi", 400, None, 101)
]

columns = [
    "user_id",
    "username",
    "email",
    "transaction_date",
    "age",
    "subscription",
    "region",
    "product_category",
    "city",
    "price",
    "status",
    "store_id"
]

df = spark.createDataFrame(data, columns)

df.show()

+-------+--------+-----------------+----------------+---+------------+------+----------------+-------+-----+--------+--------+
|user_id|username|            email|transaction_date|age|subscription|region|product_category|   city|price|  status|store_id|
+-------+--------+-----------------+----------------+---+------------+------+----------------+-------+-----+--------+--------+
|      1|   Alice|  alice@gmail.com|      2024-07-01| 25|     Premium|  West|     Electronics|  Delhi|  500|  Active|     101|
|      2|     Bob|             NULL|      2024-07-01| 30|       Basic|  East|        Clothing| Mumbai|  200|    NULL|     102|
|      3| Charlie|charlie@gmail.com|      2024-07-02| 22|     Premium|  West|     Electronics|  Delhi|  700|  Active|     101|
|      4|   David|  david@gmail.com|      2024-07-02| 35|     Premium| North|       Furniture|Chennai| NULL|Inactive|     103|
|      5|     Eva|                 |      2024-07-03| 28|       Basic|  West|        Clothing|  Delhi|  400|   

## Spark DataFrame

A Spark DataFrame is a distributed collection of data organized into named columns. It provides a tabular structure similar to a database table or pandas DataFrame and supports efficient distributed processing.

## Q1. What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

### Answer

Traditional MapReduce has several limitations that make it less suitable for modern big data processing:

- It relies heavily on disk I/O, making processing slower.
- It is not efficient for iterative algorithms such as machine learning.
- It requires writing intermediate results to disk after every stage.
- Programming is more complex due to extensive boilerplate code.
- It has higher latency for interactive data analysis.

Apache Spark overcomes these limitations by using in-memory computation, providing faster processing, simpler APIs, support for multiple programming languages, and built-in libraries for SQL, Machine Learning, Streaming, and Graph Processing.

## Q2. Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

### Answer

Spark stores intermediate data in memory (RAM) instead of repeatedly writing it to disk. Since iterative machine learning algorithms process the same data multiple times, keeping the data in memory eliminates repeated disk I/O, significantly improving execution speed and reducing latency compared to traditional disk-based systems like MapReduce.

## Q3. Remove duplicate rows based on user_id and transaction_date.

In [4]:
df_q3 = df.dropDuplicates(["user_id", "transaction_date"])

print("Q3 Output")
df_q3.show()

Q3 Output


+-------+--------+-----------------+----------------+---+------------+------+----------------+-------+-----+--------+--------+
|user_id|username|            email|transaction_date|age|subscription|region|product_category|   city|price|  status|store_id|
+-------+--------+-----------------+----------------+---+------------+------+----------------+-------+-----+--------+--------+
|      1|   Alice|  alice@gmail.com|      2024-07-01| 25|     Premium|  West|     Electronics|  Delhi|  500|  Active|     101|
|      2|     Bob|             NULL|      2024-07-01| 30|       Basic|  East|        Clothing| Mumbai|  200|    NULL|     102|
|      3| Charlie|charlie@gmail.com|      2024-07-02| 22|     Premium|  West|     Electronics|  Delhi|  700|  Active|     101|
|      4|   David|  david@gmail.com|      2024-07-02| 35|     Premium| North|       Furniture|Chennai| NULL|Inactive|     103|
|      5|     Eva|                 |      2024-07-03| 28|       Basic|  West|        Clothing|  Delhi|  400|   

### Explanation

The `dropDuplicates()` function removes duplicate rows based on the specified columns. Here, duplicate records are identified using `user_id` and `transaction_date`, ensuring that only one unique record for each combination is retained.

### Q4. Given a DataFrame `df_sales`, write a query to filter for rows where the region is `'West'` and then group by `product_category` to find the average `sale_amount`.



In [5]:
from pyspark.sql.functions import avg

df_q4 = (
    df.filter(df.region == "West")
      .groupBy("product_category")
      .agg(avg("price").alias("Average_Sale_Amount"))
)

print("Q4 Output")
df_q4.show()

Q4 Output
+----------------+-------------------+
|product_category|Average_Sale_Amount|
+----------------+-------------------+
|     Electronics|              600.0|
|        Clothing|              400.0|
+----------------+-------------------+



### Explanation

The DataFrame is first filtered to include only records where the **region** is **'West'**. The filtered data is then grouped by **product_category**, and the average of the **price** column (used as the sales amount in this sample dataset) is calculated using the `avg()` aggregation function.

## Q5. What is the difference between `.na.drop()` and `.na.fill()`? Provide a code example of filling null values in a `status` column with the string `'Unknown'`.

### Answer

- **`.na.drop()`** removes rows that contain null values.
- **`.na.fill()`** replaces null values with a specified value without removing the rows.

In this example, all null values in the `status` column are replaced with `"Unknown"`.

In [6]:
df_q5 = df.na.fill({"status": "Unknown"})

print("Q5 Output")
df_q5.select("user_id", "username", "status").show()

Q5 Output
+-------+--------+--------+
|user_id|username|  status|
+-------+--------+--------+
|      1|   Alice|  Active|
|      2|     Bob| Unknown|
|      3| Charlie|  Active|
|      4|   David|Inactive|
|      5|     Eva| Unknown|
|      5|     Eva| Unknown|
+-------+--------+--------+



## Q6. Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

### Answer

The DataFrame is grouped by the `city` column, the number of records is counted, and only cities with more than 100 records are displayed.

In [7]:
from pyspark.sql.functions import count

df_q6 = (
    df.groupBy("city")
      .agg(count("*").alias("Total_Records"))
      .filter("Total_Records > 100")
)

print("Q6 Output")
df_q6.show()

Q6 Output
+----+-------------+
|city|Total_Records|
+----+-------------+
+----+-------------+



## Q7. How does the immutability of Spark DataFrames affect how you perform data cleaning steps like dropping columns or renaming them?

### Answer

Spark DataFrames are immutable, meaning they cannot be modified after creation. Operations such as dropping columns, renaming columns, or filtering data create a new DataFrame instead of modifying the existing one. This ensures data consistency and allows Spark to optimize execution using its query planner.

## Q8. Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

In [8]:
df_q8 = df.filter(
    (df.age.between(18, 30)) &
    (df.subscription == "Premium")
)

print("Q8 Output")
df_q8.show()

Q8 Output
+-------+--------+-----------------+----------------+---+------------+------+----------------+-----+-----+------+--------+
|user_id|username|            email|transaction_date|age|subscription|region|product_category| city|price|status|store_id|
+-------+--------+-----------------+----------------+---+------------+------+----------------+-----+-----+------+--------+
|      1|   Alice|  alice@gmail.com|      2024-07-01| 25|     Premium|  West|     Electronics|Delhi|  500|Active|     101|
|      3| Charlie|charlie@gmail.com|      2024-07-02| 22|     Premium|  West|     Electronics|Delhi|  700|Active|     101|
+-------+--------+-----------------+----------------+---+------------+------+----------------+-----+-----+------+--------+



## Q9. When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?

### Answer

Handling null values before performing aggregations ensures accurate calculations. Null values may lead to incorrect or incomplete results if ignored. Replacing or removing null values improves data quality and ensures reliable statistical analysis.

## Q10. Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.

In [9]:
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType

df_q10 = (
    df.withColumn(
        "event_time",
        col("transaction_date").cast(TimestampType())
    )
    .drop("transaction_date")
)

print("Q10 Output")
df_q10.show()

Q10 Output
+-------+--------+-----------------+---+------------+------+----------------+-------+-----+--------+--------+-------------------+
|user_id|username|            email|age|subscription|region|product_category|   city|price|  status|store_id|         event_time|
+-------+--------+-----------------+---+------------+------+----------------+-------+-----+--------+--------+-------------------+
|      1|   Alice|  alice@gmail.com| 25|     Premium|  West|     Electronics|  Delhi|  500|  Active|     101|2024-07-01 00:00:00|
|      2|     Bob|             NULL| 30|       Basic|  East|        Clothing| Mumbai|  200|    NULL|     102|2024-07-01 00:00:00|
|      3| Charlie|charlie@gmail.com| 22|     Premium|  West|     Electronics|  Delhi|  700|  Active|     101|2024-07-02 00:00:00|
|      4|   David|  david@gmail.com| 35|     Premium| North|       Furniture|Chennai| NULL|Inactive|     103|2024-07-02 00:00:00|
|      5|     Eva|                 | 28|       Basic|  West|        Clothing|  

## Q11. Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

### Answer

A shuffle is the process of redistributing data across partitions during operations such as `groupBy()`, `join()`, or `distinct()`. It is considered a **wide transformation** because data from multiple partitions is exchanged across the cluster. Shuffling increases network communication and disk I/O, making these operations more expensive than narrow transformations.

## Q12. Write a code snippet that identifies and removes rows where the `email` column contains null values OR the `username` is an empty string.

In [10]:
df_q12 = df.filter(
    (df.email.isNotNull()) &
    (df.username != "")
)

print("Q12 Output")
df_q12.show()

Q12 Output
+-------+--------+-----------------+----------------+---+------------+------+----------------+-------+-----+--------+--------+
|user_id|username|            email|transaction_date|age|subscription|region|product_category|   city|price|  status|store_id|
+-------+--------+-----------------+----------------+---+------------+------+----------------+-------+-----+--------+--------+
|      1|   Alice|  alice@gmail.com|      2024-07-01| 25|     Premium|  West|     Electronics|  Delhi|  500|  Active|     101|
|      3| Charlie|charlie@gmail.com|      2024-07-02| 22|     Premium|  West|     Electronics|  Delhi|  700|  Active|     101|
|      4|   David|  david@gmail.com|      2024-07-02| 35|     Premium| North|       Furniture|Chennai| NULL|Inactive|     103|
|      5|     Eva|                 |      2024-07-03| 28|       Basic|  West|        Clothing|  Delhi|  400|    NULL|     101|
|      5|     Eva|                 |      2024-07-03| 28|       Basic|  West|        Clothing|  Delh

## Q13. How do you use the `.agg()` function to calculate multiple statistics at once, such as the min, max, and mean of the `price` column?

In [11]:
from pyspark.sql.functions import min, max, avg

df_q13 = df.agg(
    min("price").alias("Minimum Price"),
    max("price").alias("Maximum Price"),
    avg("price").alias("Average Price")
)

print("Q13 Output")
df_q13.show()

Q13 Output
+-------------+-------------+-------------+
|Minimum Price|Maximum Price|Average Price|
+-------------+-------------+-------------+
|          200|          700|        440.0|
+-------------+-------------+-------------+



## Q14. In the context of cleaning a dataset, what is the risk of using `inferSchema=true` when your source data contains messy or inconsistent date formats?

### Answer

When `inferSchema=true` is used on data with inconsistent or messy date formats, Spark may incorrectly infer the data type. This can result in dates being interpreted as strings, null values, or incorrect timestamps. It is generally recommended to define the schema explicitly for better accuracy and consistency.

## Q15. Write a final processing pipeline that:

1. Filters out duplicates.
2. Fills null prices with 0.
3. Groups by `store_id` to calculate total revenue.

In [12]:
from pyspark.sql.functions import sum

df_q15 = (
    df.dropDuplicates()
      .na.fill({"price": 0})
      .groupBy("store_id")
      .agg(sum("price").alias("Total Revenue"))
)

print("Q15 Output")
df_q15.show()

Q15 Output
+--------+-------------+
|store_id|Total Revenue|
+--------+-------------+
|     103|            0|
|     101|         1600|
|     102|          200|
+--------+-------------+

